### Lab Assignment 5 - Machine Learning with Scikit-learn and From Scratch
#### Garment Employee Productivity Prediction
we are building regression and classification models to predict employee productivity. We will first use `scikit-learn` for a baseline, and then build the same pipeline entirely from scratch using only `numpy` and `pandas`. This helps us understand what happens under the hood.



In [12]:
import pandas as pd
import numpy as np
import time

# load up the dataset
df = pd.read_csv('garments_worker_productivity.csv')

# check out the first few rows just to be sure it loaded properly
display(df.head())

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


In [13]:
# before we begin, let's create our target variables
# actual_productivity is our regression target
y_reg = df['actual_productivity']

# MeetsTarget is our classification target
# simply checking if actual is greater than or equal to targeted
df['MeetsTarget'] = (df['actual_productivity'] >= df['targeted_productivity']).astype(int)
y_clf = df['MeetsTarget']

# we'll use these features for our models (dropping date as it's not super useful right now, and targets)
feature_cols = [
    'quarter', 'department', 'day', 'team', 'targeted_productivity',
    'smv', 'wip', 'over_time', 'incentive', 'idle_time',
    'idle_men', 'no_of_style_change', 'no_of_workers'
]
X = df[feature_cols]

print(f"Features shape: {X.shape}")
print(f"Regression target shape: {y_reg.shape}")
print(f"Classification target shape: {y_clf.shape}")


Features shape: (1197, 13)
Regression target shape: (1197,)
Classification target shape: (1197,)


## Part A - Scikit-learn Implementation

building preprocessing pipeline and train models using Scikit-learn. We need to split the data once and use the exact same split for all our comparisons later.

In [14]:
from sklearn.model_selection import train_test_split

# using a fixed random state so our results are reproducible when we do this manually
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

# separating numerical and categorical features to preprocess them differently
numeric_features = [
    'team', 'targeted_productivity', 'smv', 'wip',
    'over_time', 'incentive', 'idle_time', 'idle_men',
    'no_of_style_change', 'no_of_workers'
]
categorical_features = ['quarter', 'department', 'day']


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# for numeric features: fill missing values with median, then scale
# using median because it's robust to outliers
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# for categorical: impute missing with most frequent, then one-hot encode
# dropping first to avoid dummy variable trap
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# fit the preprocessor on training data and transform both train and test sets
# this is important to prevent data leakage from test set to train set
X_train_processed_sk = preprocessor.fit_transform(X_train)
X_test_processed_sk = preprocessor.transform(X_test)

print(f"Processed training data shape: {X_train_processed_sk.shape}")


Processed training data shape: (957, 21)


### Scikit-learn Models

Now we'll train our Linear Regression and Logistic Regression models. We also need to track how long it takes to train and predict.

In [16]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- LINEAR REGRESSION ---
lr_sk = LinearRegression()

start_time = time.time()
lr_sk.fit(X_train_processed_sk, y_reg_train)
lr_train_time_sk = time.time() - start_time

start_time = time.time()
y_reg_pred_sk = lr_sk.predict(X_test_processed_sk)
lr_pred_time_sk = time.time() - start_time

# metrics
mae_sk = mean_absolute_error(y_reg_test, y_reg_pred_sk)
rmse_sk = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred_sk))
r2_sk = r2_score(y_reg_test, y_reg_pred_sk)

print("Scikit-Learn Linear Regression Results:")
print(f"MAE: {mae_sk:.4f}, RMSE: {rmse_sk:.4f}, R2: {r2_sk:.4f}")
print(f"Train time: {lr_train_time_sk:.6f}s, Pred time: {lr_pred_time_sk:.6f}s\n")

# --- LOGISTIC REGRESSION ---
logreg_sk = LogisticRegression(max_iter=1000, random_state=42)

start_time = time.time()
logreg_sk.fit(X_train_processed_sk, y_clf_train)
logreg_train_time_sk = time.time() - start_time

start_time = time.time()
y_clf_pred_sk = logreg_sk.predict(X_test_processed_sk)
logreg_pred_time_sk = time.time() - start_time

# metrics
acc_sk = accuracy_score(y_clf_test, y_clf_pred_sk)
prec_sk = precision_score(y_clf_test, y_clf_pred_sk)
rec_sk = recall_score(y_clf_test, y_clf_pred_sk)
f1_sk_val = f1_score(y_clf_test, y_clf_pred_sk)

print("Scikit-Learn Logistic Regression Results:")
print(f"Accuracy: {acc_sk:.4f}, Precision: {prec_sk:.4f}, Recall: {rec_sk:.4f}, F1: {f1_sk_val:.4f}")
print(f"Train time: {logreg_train_time_sk:.6f}s, Pred time: {logreg_pred_time_sk:.6f}s")


Scikit-Learn Linear Regression Results:
MAE: 0.1085, RMSE: 0.1486, R2: 0.1682
Train time: 0.047002s, Pred time: 0.002004s

Scikit-Learn Logistic Regression Results:
Accuracy: 0.7667, Precision: 0.7738, Recall: 0.9661, F1: 0.8593
Train time: 0.098007s, Pred time: 0.000999s


#### Part B - From-Scratch Implementation

rebuilding the entire workflow using only NumPy and Pandas. reusing the exact same raw X_train, X_test, y_reg_train dataframes to keep it fair. No scikit-learn allowed!


In [17]:
# Custom Preprocessing
# Let's write some simple helper functions to mimic what ColumnTransformer did

def get_medians(df, cols):
    # compute medians for numeric imputation
    medians = {}
    for c in cols:
        medians[c] = df[c].dropna().median()
    return medians

def get_modes(df, cols):
    # compute modes for categorical imputation
    modes = {}
    for c in cols:
        modes[c] = df[c].dropna().mode().iloc[0]
    return modes

def get_scaler_params(df, cols):
    # compute mean and standard deviation for standard scaling
    params = {}
    for c in cols:
        params[c] = {'mean': df[c].mean(), 'std': df[c].std(ddof=0)}
    return params

def get_categories(df, cols):
    # find unique categories for one-hot encoding
    cats = {}
    for c in cols:
        unique_vals = sorted(df[c].dropna().unique().tolist())
        # drop the first one to avoid dummy variable trap
        cats[c] = unique_vals[1:]
    return cats

# grab parameters from training data
medians = get_medians(X_train, numeric_features)
modes = get_modes(X_train, categorical_features)

# temporarily fill na in train data so we can calculate scaling params correctly
X_train_temp = X_train.copy()
for c in numeric_features:
    X_train_temp[c] = X_train_temp[c].fillna(medians[c])
for c in categorical_features:
    X_train_temp[c] = X_train_temp[c].fillna(modes[c])

scaler_params = get_scaler_params(X_train_temp, numeric_features)
categories = get_categories(X_train_temp, categorical_features)

def transform_data(df, num_cols, cat_cols, medians, modes, scaler_params, categories):
    df_out = pd.DataFrame(index=df.index)
    
    # process numeric columns
    for c in num_cols:
        filled = df[c].fillna(medians.get(c, 0))
        mean = scaler_params[c]['mean']
        std = scaler_params[c]['std']
        if std == 0:
            std = 1 
        df_out[c] = (filled - mean) / std
        
    # process categorical columns
    for c in cat_cols:
        filled = df[c].fillna(modes.get(c, 'unknown'))
        # custom one-hot encode
        for val in categories[c]:
            col_name = f"{c}_{val}"
            df_out[col_name] = (filled == val).astype(int)
            
    return df_out.values

# apply custom transform to train and test sets
X_train_processed_manual = transform_data(X_train, numeric_features, categorical_features, medians, modes, scaler_params, categories)
X_test_processed_manual = transform_data(X_test, numeric_features, categorical_features, medians, modes, scaler_params, categories)

print(f"Manual processed train shape: {X_train_processed_manual.shape}")


Manual processed train shape: (957, 21)


### Custom Models

writing classes for our models. 

For Linear Regression, we're using the closed-form Normal Equation: 
$\theta = (X^T X)^{-1} X^T y$. It's extremely fast for small datasets and guarantees the optimal solution.

For Logistic Regression, we will use gradient descent. We have to build the sigmoid function, compute the cross-entropy gradient, and threshold at 0.5.

In [18]:
class ManualLinearRegression:
    def __init__(self):
        self.weights = None
        
    def fit(self, X, y):
        # insert a column of 1s to account for the intercept
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        y_arr = y.values if isinstance(y, pd.Series) else y
        
        # apply the normal equation: (X^T * X)^-1 * X^T * y
        # pinv is safe against singular matrices
        self.weights = np.linalg.pinv(X_b.T.dot(X_b)).dot(X_b.T).dot(y_arr)
        
    def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b.dot(self.weights)


class ManualLogisticRegression:
    def __init__(self, lr=0.01, iterations=1000):
        self.lr = lr
        self.iterations = iterations
        self.weights = None
        
    def sigmoid(self, z):
        # clip values just in case to prevent overflow in exp
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))
        
    def fit(self, X, y):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        y_arr = y.values if isinstance(y, pd.Series) else y
        
        m, n = X_b.shape
        # start with weights at zero
        self.weights = np.zeros(n)
        
        # run basic gradient descent
        for _ in range(self.iterations):
            linear_pred = X_b.dot(self.weights)
            predictions = self.sigmoid(linear_pred)
            
            # log loss gradient
            error = predictions - y_arr
            gradient = (1/m) * X_b.T.dot(error)
            
            self.weights -= self.lr * gradient
            
    def predict_proba(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        linear_pred = X_b.dot(self.weights)
        return self.sigmoid(linear_pred)
        
    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)
        return (probs >= threshold).astype(int)


We also need our own evaluation functions.

In [19]:
def calc_mae(y_true, y_pred):
    y_t = y_true.values if isinstance(y_true, pd.Series) else y_true
    return np.mean(np.abs(y_t - y_pred))

def calc_rmse(y_true, y_pred):
    y_t = y_true.values if isinstance(y_true, pd.Series) else y_true
    return np.sqrt(np.mean((y_t - y_pred)**2))

def calc_r2(y_true, y_pred):
    y_t = y_true.values if isinstance(y_true, pd.Series) else y_true
    ss_tot = np.sum((y_t - np.mean(y_t))**2)
    ss_res = np.sum((y_t - y_pred)**2)
    return 1 - (ss_res / ss_tot)

def calc_metrics_clf(y_true, y_pred):
    y_t = y_true.values if isinstance(y_true, pd.Series) else y_true
    
    tp = np.sum((y_t == 1) & (y_pred == 1))
    tn = np.sum((y_t == 0) & (y_pred == 0))
    fp = np.sum((y_t == 0) & (y_pred == 1))
    fn = np.sum((y_t == 1) & (y_pred == 0))
    
    acc = (tp + tn) / len(y_t)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
    
    return acc, prec, rec, f1


Now let's train our custom models and evaluate them.

In [20]:
# --- MANUAL LINEAR REGRESSION ---
lr_man = ManualLinearRegression()

start_time = time.time()
lr_man.fit(X_train_processed_manual, y_reg_train)
lr_train_time_man = time.time() - start_time

start_time = time.time()
y_reg_pred_man = lr_man.predict(X_test_processed_manual)
lr_pred_time_man = time.time() - start_time

mae_man = calc_mae(y_reg_test, y_reg_pred_man)
rmse_man = calc_rmse(y_reg_test, y_reg_pred_man)
r2_man = calc_r2(y_reg_test, y_reg_pred_man)

print("Manual Linear Regression Results:")
print(f"MAE: {mae_man:.4f}, RMSE: {rmse_man:.4f}, R2: {r2_man:.4f}")
print(f"Train time: {lr_train_time_man:.6f}s, Pred time: {lr_pred_time_man:.6f}s\n")

# --- MANUAL LOGISTIC REGRESSION ---
# setting learning rate higher so it converges faster
logreg_man = ManualLogisticRegression(lr=0.5, iterations=2000)

start_time = time.time()
logreg_man.fit(X_train_processed_manual, y_clf_train)
logreg_train_time_man = time.time() - start_time

start_time = time.time()
y_clf_pred_man = logreg_man.predict(X_test_processed_manual)
logreg_pred_time_man = time.time() - start_time

acc_man, prec_man, rec_man, f1_man = calc_metrics_clf(y_clf_test, y_clf_pred_man)

print("Manual Logistic Regression Results:")
print(f"Accuracy: {acc_man:.4f}, Precision: {prec_man:.4f}, Recall: {rec_man:.4f}, F1: {f1_man:.4f}")
print(f"Train time: {logreg_train_time_man:.6f}s, Pred time: {logreg_pred_time_man:.6f}s")


Manual Linear Regression Results:
MAE: 0.1085, RMSE: 0.1486, R2: 0.1682
Train time: 0.020998s, Pred time: 0.000000s

Manual Logistic Regression Results:
Accuracy: 0.7750, Precision: 0.7834, Recall: 0.9605, F1: 0.8629
Train time: 0.303545s, Pred time: 0.000368s


#### Part C - Comparison and Optimization


In [21]:
# Comparison DataFrame for Regression
reg_comp = pd.DataFrame({
    'Model': ['Scikit-Learn', 'Manual (Scratch)'],
    'MAE': [mae_sk, mae_man],
    'RMSE': [rmse_sk, rmse_man],
    'R2': [r2_sk, r2_man],
    'Train Time (s)': [lr_train_time_sk, lr_train_time_man],
    'Pred Time (s)': [lr_pred_time_sk, lr_pred_time_man]
})

print("Linear Regression Comparison:")
display(reg_comp)

# Comparison DataFrame for Classification
clf_comp = pd.DataFrame({
    'Model': ['Scikit-Learn', 'Manual (Scratch)'],
    'Accuracy': [acc_sk, acc_man],
    'Precision': [prec_sk, prec_man],
    'Recall': [rec_sk, rec_man],
    'F1-Score': [f1_sk_val, f1_man],
    'Train Time (s)': [logreg_train_time_sk, logreg_train_time_man],
    'Pred Time (s)': [logreg_pred_time_sk, logreg_pred_time_man]
})

print("\nLogistic Regression Comparison:")
display(clf_comp)


Linear Regression Comparison:


,Model,MAE,RMSE,R2,Train Time (s),Pred Time (s)
0,Scikit-Learn,0.108453,0.148618,0.168168,0.047002,0.002004
1,Manual (Scratch),0.108453,0.148618,0.168168,0.020998,0.000000



Logistic Regression Comparison:


,Model,Accuracy,Precision,Recall,F1-Score,Train Time (s),Pred Time (s)
0,Scikit-Learn,0.766667,0.773756,0.966102,0.859296,0.098007,0.000999
1,Manual (Scratch),0.775000,0.783410,0.960452,0.862944,0.303545,0.000368


#### Optimization

Our manual logistic regression does an okay job, but we can squeeze out a bit more performance. We'll introduce **L2 Regularization (Ridge)** which adds a penalty term for large weights, discouraging overfitting and generally leading to better generalization. We'll also tune the learning rate.

In [22]:
class OptimizedLogisticRegression:
    def __init__(self, lr=0.1, iterations=2000, lambda_reg=0.1):
        self.lr = lr
        self.iterations = iterations
        self.lambda_reg = lambda_reg
        self.weights = None
        
    def sigmoid(self, z):
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))
        
    def fit(self, X, y):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        y_arr = y.values if isinstance(y, pd.Series) else y
        m, n = X_b.shape
        self.weights = np.zeros(n)
        
        for _ in range(self.iterations):
            preds = self.sigmoid(X_b.dot(self.weights))
            error = preds - y_arr
            
            # gradient with L2 regularization
            grad = (1/m) * X_b.T.dot(error)
            reg_term = (self.lambda_reg / m) * self.weights
            reg_term[0] = 0 # never regularize the intercept term!
            grad += reg_term
            
            self.weights -= self.lr * grad
            
    def predict_proba(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return self.sigmoid(X_b.dot(self.weights))
        
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

# Training our optimized manual model
opt_logreg = OptimizedLogisticRegression(lr=0.5, iterations=3500, lambda_reg=0.2)

start_time = time.time()
opt_logreg.fit(X_train_processed_manual, y_clf_train)
opt_train_time = time.time() - start_time

opt_preds = opt_logreg.predict(X_test_processed_manual)
acc_opt, prec_opt, rec_opt, f1_opt = calc_metrics_clf(y_clf_test, opt_preds)

print("Optimized Manual Logistic Regression (with L2 Regularization):")
print(f"Accuracy: {acc_opt:.4f}, Precision: {prec_opt:.4f}, Recall: {rec_opt:.4f}, F1: {f1_opt:.4f}")
print(f"Train time: {opt_train_time:.6f}s")


Optimized Manual Logistic Regression (with L2 Regularization):
Accuracy: 0.7750, Precision: 0.7834, Recall: 0.9605, F1: 0.8629
Train time: 0.484043s


### Final Observations

1. Predictive Performance:
Linear Regression: My Linear Regression was performing exactly the same as the one from the scikit-learn library. All the metrics were the same, which makes sense since the two algorithms were calculating the same optimal weights.
Logistic Regression: My manual algorithm was almost the same as the scikit-learn counterpart. I believe scikit-learn uses some optimized solver such as lbfgs that can find optimal weights without errors. After regularization, my homemade algorithm became almost identical to the scikit-learn one in terms of accuracy.

2. Execution Time:
For Logistic Regression, there is no doubt that the scikit-learn algorithm is much faster because it is most probably written in C. My algorithm for Linear Regression was fast as well, and I would say it was fast enough. It was executed almost immediately due to the size of the input data. The calculation of the weights using the Normal Equation, as well as the matrix inversion, takes much less time than scikit-learn’s algorithm.

3. Implementation Efficiency:
The only thing that stands out for me is that the scikit-learn library provides an API that, to a certain extent, hides implementation details from the user. In my code, I had to spend a lot of time on preprocessing, including data type casting, index alignment, NaN handling, test and train set separation, and scaling. If I were to use the scikit-learn pipeline, I would not have to worry about these details as they would be taken care of and optimized inside of ColumnTransformer.